# Marlin-2B Web UI
### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
###

Deploy `lunahr/Marlin-2B-ungated` with a public Pinggy tunnel.

- **Hardware:** NVIDIA T4 GPU (15GB VRAM)
- **Optimizations:** `uv` installation, `ffmpeg` & `PyAV` video fallback, T4-compatible attention.

In [ ]:
#@title Initialize Environment & Model { display-mode: "form" }
import subprocess
import sys
import os
import time
from IPython.display import display, HTML

def stream_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
    return process.poll()

if not os.path.exists(".setup_complete_v4"):
    print("--> [1/4] Installing System Dependencies (ffmpeg, aria2c & ssh client)...")
    stream_command("apt-get update && apt-get install -y aria2 openssh-client ffmpeg libsm6 libxext6")

    print("\n--> [2/4] Deploying UV Package Manager for rapid resolution...")
    stream_command("pip install uv")

    print("\n--> [3/4] Synchronizing Python Backend...")
    stream_command("uv pip install --system 'transformers>=5.7.0' av decord qwen-vl-utils pillow accelerate flask werkzeug")

    print("\n[ALERT] Core system libraries updated. Restarting Colab runtime to flush memory...")
    print("⚠️ Please run this cell ONE MORE TIME after the session automatically restarts. ⚠️")

    with open(".setup_complete_v4", "w") as f:
        f.write("done")
    time.sleep(2)
    os.kill(os.getpid(), 9)
else:
    print("--> [✓] Environment setup detected. Bypassing installation phase.")

print("\n--> [4/4] Loading Marlin-2B Model Pipeline onto T4 GPU...")
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

model_id = "lunahr/Marlin-2B-ungated"

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

animated_html = """
<div style="padding:15px; border-radius:10px; background: linear-gradient(135deg, #1e1e2f 0%, #2a2a40 100%); color:#00ffcc; font-family:monospace; box-shadow: 0 4px 15px rgba(0,0,0,0.5); animation: pulse 2s infinite alternate;">
    <h3 style="color:#ff007f; margin-top:0;">✨ Initialization Complete ✨</h3>
    <p>System dependencies, stable video decoders, and the Marlin-2B model are successfully loaded into VRAM.</p>
</div>
<style>@keyframes pulse { 0% { box-shadow: 0 4px 15px rgba(0,255,204,0.2); } 100% { box-shadow: 0 4px 25px rgba(255,0,127,0.4); } }</style>
"""
display(HTML(animated_html))

In [ ]:
#@title Launch Custom Web UI & Pinggy Tunnel { display-mode: "form" }
import os
import subprocess
import time
import re
import logging
from IPython.display import display, HTML
from flask import Flask, request, jsonify, render_template_string, make_response
from werkzeug.utils import secure_filename

# 0. Environment Cleanup (Prevents Port Collisions on Reruns)
os.system("pkill -f ssh")

# 1. Initialize Custom Web Server Framework
app = Flask(__name__)
UPLOAD_FOLDER = 'marlin_workspace'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

# 2. Premium Frontend Architecture (HTML/CSS/JS)
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Marlin-2B Vision Intelligence</title>
    <style>
        @keyframes gradientBG {
            0% { background-position: 0% 50%; }
            50% { background-position: 100% 50%; }
            100% { background-position: 0% 50%; }
        }
        @keyframes pulseGlow {
            0% { box-shadow: 0 0 15px rgba(0, 255, 204, 0.2); }
            100% { box-shadow: 0 0 25px rgba(255, 0, 127, 0.5); }
        }
        @keyframes slideIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }
        body {
            margin: 0;
            padding: 0;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(-45deg, #0f0c29, #302b63, #24243e);
            background-size: 400% 400%;
            animation: gradientBG 15s ease infinite;
            color: #ffffff;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
        }
        .container {
            background: rgba(25, 25, 35, 0.85);
            backdrop-filter: blur(10px);
            border-radius: 16px;
            padding: 40px;
            width: 95%;
            max-width: 1100px;
            animation: pulseGlow 3s infinite alternate, slideIn 0.8s ease-out forwards;
            border: 1px solid rgba(255, 255, 255, 0.1);
        }
        h1 {
            text-align: center;
            color: #00ffcc;
            text-shadow: 0 0 10px rgba(0, 255, 204, 0.5);
            margin-top: 0;
            margin-bottom: 30px;
        }
        .grid {
            display: grid;
            grid-template-columns: 1fr 1.2fr;
            gap: 30px;
        }
        @media (max-width: 768px) { .grid { grid-template-columns: 1fr; } }
        .form-group {
            margin-bottom: 20px;
        }
        label {
            display: block;
            margin-bottom: 8px;
            font-weight: bold;
            color: #e0e0e0;
        }
        input[type="file"], input[type="text"], select {
            width: 100%;
            padding: 12px;
            border-radius: 8px;
            border: 1px solid #444;
            background: #1a1a2e;
            color: white;
            box-sizing: border-box;
            transition: border-color 0.3s;
        }
        input[type="text"]:focus, select:focus {
            border-color: #00ffcc;
            outline: none;
        }
        button[type="submit"] {
            width: 100%;
            padding: 15px;
            background: linear-gradient(90deg, #ff007f, #7000ff);
            color: white;
            border: none;
            border-radius: 8px;
            font-size: 16px;
            font-weight: bold;
            cursor: pointer;
            transition: transform 0.2s, box-shadow 0.2s;
        }
        button[type="submit"]:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(255, 0, 127, 0.4);
        }
        button[type="submit"]:disabled {
            background: #555;
            cursor: not-allowed;
            transform: none;
            box-shadow: none;
        }
        .output-container {
            position: relative;
            margin-bottom: 20px;
        }
        .copy-btn {
            position: absolute;
            top: 10px;
            right: 10px;
            background: rgba(0, 255, 204, 0.1);
            border: 1px solid #00ffcc;
            color: #00ffcc;
            padding: 6px 12px;
            border-radius: 6px;
            font-size: 12px;
            font-weight: bold;
            cursor: pointer;
            transition: all 0.2s ease;
            z-index: 10;
        }
        .copy-btn:hover {
            background: #00ffcc;
            color: #111;
            box-shadow: 0 0 10px rgba(0, 255, 204, 0.5);
        }
        .output-box {
            background: #0a0a10;
            border-radius: 8px;
            padding: 20px;
            padding-top: 45px;
            height: 180px;
            overflow-y: auto;
            border: 1px solid #333;
            font-family: monospace;
            color: #ff007f;
            white-space: pre-wrap;
        }
        .text-box {
            background: #151525;
            border-radius: 8px;
            padding: 20px;
            padding-top: 45px;
            height: 180px;
            overflow-y: auto;
            border: 1px solid #444;
            color: #e0e0e0;
            white-space: pre-wrap;
            font-family: 'Segoe UI', Tahoma, sans-serif;
            font-size: 15px;
            line-height: 1.6;
        }
        .loader {
            display: none;
            text-align: center;
            color: #ff007f;
            font-weight: bold;
            margin-top: 15px;
            animation: pulseGlow 1s infinite alternate;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>✨ Marlin-2B Interactive Hub ✨</h1>
        <div class="grid">
            <div class="controls">
                <form id="inferenceForm" onsubmit="executePipeline(event)">
                    <div class="form-group">
                        <label for="videoFile">Upload Target Video Asset (.mp4)</label>
                        <input type="file" id="videoFile" name="video" accept="video/mp4" required>
                    </div>
                    <div class="form-group">
                        <label for="mode">Execution Mode</label>
                        <select id="mode" name="mode">
                            <option value="Dense Captioning">Dense Captioning</option>
                            <option value="Temporal Grounding">Temporal Grounding / Search</option>
                        </select>
                    </div>
                    <div class="form-group">
                        <label for="query">Search Query (For Grounding Mode)</label>
                        <input type="text" id="query" name="query" placeholder="e.g. A person is walking...">
                    </div>
                    <button type="submit" id="submitBtn">Execute Multimodal Pipeline</button>
                    <div class="loader" id="loader">Processing visual tensor data... Please wait.</div>
                </form>
            </div>

            <div class="results">
                <label>Raw JSON Stream</label>
                <div class="output-container">
                    <button class="copy-btn" onclick="copyText('outputBox', this)">Copy JSON</button>
                    <div class="output-box" id="outputBox">{ "status": "Waiting for input..." }</div>
                </div>

                <label>Plain Text Output</label>
                <div class="output-container">
                    <button class="copy-btn" onclick="copyText('plainTextBox', this)">Copy Text</button>
                    <div class="text-box" id="plainTextBox">Waiting for input...</div>
                </div>
            </div>
        </div>
    </div>

    <script>
        function copyText(elementId, btnElement) {
            const textToCopy = document.getElementById(elementId).innerText;
            navigator.clipboard.writeText(textToCopy).then(() => {
                const originalText = btnElement.innerText;
                btnElement.innerText = 'Copied!';
                setTimeout(() => {
                    btnElement.innerText = originalText;
                }, 2000);
            }).catch(err => {
                console.error('Failed to copy text: ', err);
            });
        }

        function extractPlainText(data) {
            if (typeof data === 'string') return data;
            if (Array.isArray(data)) return data.map(item => extractPlainText(item)).join('\\n\\n');
            if (typeof data === 'object' && data !== null) {
                let parts = [];
                for (const [key, value] of Object.entries(data)) {
                    parts.push(`${key.charAt(0).toUpperCase() + key.slice(1)}: ${extractPlainText(value)}`);
                }
                return parts.join('\\n');
            }
            return String(data);
        }

        async function executePipeline(event) {
            // CRITICAL: Physically blocks the browser from reloading the page
            event.preventDefault();

            const form = event.target;
            const videoInput = document.getElementById('videoFile');

            if (!videoInput.files || videoInput.files.length === 0) {
                alert("Please select a video file first.");
                return;
            }

            const fileSizeMB = videoInput.files[0].size / (1024 * 1024);
            if (fileSizeMB > 50) {
                alert("File is too large! Please keep videos under 50MB for Colab Free Tier constraints.");
                return;
            }

            const formData = new FormData(form);
            const submitBtn = document.getElementById('submitBtn');
            const loader = document.getElementById('loader');
            const outputBox = document.getElementById('outputBox');
            const plainTextBox = document.getElementById('plainTextBox');

            submitBtn.disabled = true;
            loader.style.display = 'block';
            outputBox.textContent = "Uploading and analyzing asset...";
            plainTextBox.textContent = "Processing...";

            try {
                const response = await fetch('/infer', {
                    method: 'POST',
                    body: formData
                });

                const result = await response.json();

                outputBox.textContent = JSON.stringify(result, null, 2);

                if (result.status === "success" && result.data) {
                    plainTextBox.textContent = extractPlainText(result.data);
                } else if (result.error || result.message) {
                    plainTextBox.textContent = `Error Encountered:\\n${result.error || result.message}`;
                } else {
                    plainTextBox.textContent = "No text payload extracted.";
                }

            } catch (error) {
                outputBox.textContent = JSON.stringify({ error: error.message }, null, 2);
                plainTextBox.textContent = `Critical Failure:\\n${error.message}`;
            } finally {
                submitBtn.disabled = false;
                loader.style.display = 'none';
            }
        }
    </script>
</body>
</html>
"""

# 3. Define Web Endpoints
@app.route('/')
def serve_ui():
    response = make_response(render_template_string(HTML_TEMPLATE))
    response.headers['Cache-Control'] = 'no-cache, no-store, must-revalidate'
    response.headers['Pragma'] = 'no-cache'
    response.headers['Expires'] = '0'
    return response

@app.route('/infer', methods=['POST'])
def process_inference():
    if 'video' not in request.files:
        return jsonify({"error": "No video payload detected."}), 400

    file = request.files['video']
    if file.filename == '':
        return jsonify({"error": "Empty file submitted."}), 400

    try:
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)

        mode = request.form.get('mode', 'Dense Captioning')
        query = request.form.get('query', '')

        print(f"\n--> [BACKEND STREAM] Processing '{filename}' in mode: {mode}")

        # Execute Marlin pipeline utilizing the native VRAM model architecture
        if mode == "Dense Captioning":
            inference_data = model.caption(filepath)
        else:
            print(f"--> [BACKEND STREAM] Target Search: '{query}'")
            inference_data = model.find(query, video_path=filepath)

        print("--> [BACKEND STREAM] Pipeline execution complete. Transmitting payload to UI.")
        return jsonify({"status": "success", "data": inference_data})

    except Exception as e:
        print(f"--> [BACKEND ERROR] {str(e)}")
        return jsonify({"status": "error", "message": str(e)}), 500

# 4. Asynchronous Pinggy Execution via File Polling
print("--> Initializing Secure Pinggy Tunnel on Port 5000...")
log_file = "pinggy_log.txt"

if os.path.exists(log_file):
    os.remove(log_file)

os.system(f"nohup ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:5000 a.pinggy.io > {log_file} 2>&1 &")

url_found = False
pinggy_url = ""

for _ in range(15):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            content = f.read()
            if "http" in content:
                urls = re.findall(r'https://[^\s]+', content)
                for url in urls:
                    if "pinggy" in url and "dashboard" not in url:
                        pinggy_url = url
                        url_found = True
                        break
            if url_found:
                break

if url_found:
    animated_ui_notice = f"""
    <div style="padding:20px; border-radius:12px; background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 100%); color:#00ffcc; text-align:center; font-family:monospace; border: 2px solid #ff007f; box-shadow: 0 4px 15px rgba(255,0,127,0.3); animation: pulse-ping 2s infinite alternate; margin-bottom: 20px;">
        <h2 style="color:#ffffff; margin:0 0 15px 0;">🌐 Public UI Ready For Access</h2>
        <p style="font-size: 16px; margin: 0; color: #aaa;">Your isolated web application is streaming live at:</p>
        <br>
        <a href="{pinggy_url}" target="_blank" style="display: inline-block; background: linear-gradient(90deg, #ff007f, #7000ff); color: #fff; padding: 12px 25px; border-radius: 8px; text-decoration: none; font-weight: bold; letter-spacing: 1px; font-size: 18px; transition: transform 0.2s;">{pinggy_url}</a>
    </div>
    <style>@keyframes pulse-ping {{ 0% {{ box-shadow: 0 4px 15px rgba(255,0,127,0.1); }} 100% {{ box-shadow: 0 4px 30px rgba(0,255,204,0.4); }} }}</style>
    """
    display(HTML(animated_ui_notice))
else:
    print("--> [WARNING] Tunnel URL extraction timed out.")
    print("--> Dumping raw Pinggy logs so you can find the link manually:")
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            print(f.read())

# 5. Ignite Flask Engine
print("--> Booting Local Flask Server...")
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

app.run(host='0.0.0.0', port=5000, threaded=True)